# REGISTERING MODEL

In [0]:
%pip install -U sentence-transformers
%restart_python

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
print(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[ 6.76568970e-02  6.34959415e-02  4.87131625e-02  7.93050006e-02
   3.74480337e-02  2.65282649e-03  3.93749699e-02 -7.09840003e-03
   5.93614392e-02  3.15370187e-02  6.00980930e-02 -5.29051758e-02
   4.06067930e-02 -2.59308852e-02  2.98428405e-02  1.12689636e-03
   7.35149086e-02 -5.03819585e-02 -1.22386619e-01  2.37028673e-02
   2.97264811e-02  4.24767695e-02  2.56337542e-02  1.99519913e-03
  -5.69190793e-02 -2.71598287e-02 -3.29036005e-02  6.60248548e-02
   1.19007103e-01 -4.58790176e-02 -7.26214573e-02 -3.25839967e-02
   5.23413867e-02  4.50552739e-02  8.25306214e-03  3.67023647e-02
  -1.39415348e-02  6.53918609e-02 -2.64272727e-02  2.06409590e-04
  -1.36643751e-02 -3.62809859e-02 -1.95043329e-02 -2.89738048e-02
   3.94269638e-02 -8.84090886e-02  2.62424978e-03  1.36714112e-02
   4.83062901e-02 -3.11565232e-02 -1.17329165e-01 -5.11690006e-02
  -8.85287151e-02 -2.18963530e-02  1.42986728e-02  4.44167964e-02
  -1.34815611e-02  7.43392110e-02  2.66382452e-02 -1.98762454e-02
   1.79190

In [0]:
import mlflow
import tempfile
from mlflow.models import infer_signature
from mlflow.pyfunc import PythonModel


class SentenceTransformerPyfunc(PythonModel):
    def load_context(self, context):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(context.artifacts["model_dir"])

    def predict(self, context, model_input, params=None):
        # Convert DataFrame/Series input to list of strings
        if hasattr(model_input, "iloc"):
            sentences = model_input.iloc[:, 0].tolist()
        elif isinstance(model_input, list):
            sentences = model_input
        else:
            sentences = list(model_input)
        return self.model.encode(sentences)


# Save the model locally for artifact reference
model_save_path = tempfile.mkdtemp()
model.save(model_save_path)

# Generate signature from actual input/output
sample_input = ["This is an example sentence"]
sample_output = model.encode(sample_input)
signature = infer_signature(sample_input, sample_output)

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        name="sentence-transformer-model",
        python_model=SentenceTransformerPyfunc(),
        artifacts={"model_dir": model_save_path},
        signature=signature,
        input_example=sample_input,
        pip_requirements=["sentence-transformers"],
        registered_model_name="att_log_anomaly_catalog.ml.all_minilm_l6_v2",
    )

print(f"Model URI: {model_info.model_uri}")
print(f"Run ID: {model_info.run_id}")

/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🔗 View Logged Model at: https://fevm-att-log-anomaly.cloud.databricks.com/ml/experiments/2664990784440270/models/m-6b2a0e9df0184227902f7e71d0064cac?o=7474654469252811
2026/08/19 01:56:21 INFO mlflow.pyfunc: Validating input example against model signature


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Registered model 'att_log_anomaly_catalog.ml.all_minilm_l6_v2' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/21 [00:00<?, ?it/s]

🔗 Created version '3' of model 'att_log_anomaly_catalog.ml.all_minilm_l6_v2': https://fevm-att-log-anomaly.cloud.databricks.com/explore/data/models/att_log_anomaly_catalog/ml/all_minilm_l6_v2/version/3?o=7474654469252811


Model URI: models:/m-6b2a0e9df0184227902f7e71d0064cac
Run ID: 0ec7a28b3ff44c69be963f2d7de8267d


In [0]:
import mlflow

loaded_model = mlflow.pyfunc.load_model("models:/att_log_anomaly_catalog.ml.all_minilm_l6_v2/3")

# Test with sample sentences
test_sentences = ["Hello world", "MLflow model loading works!"]
test_embeddings = loaded_model.predict(test_sentences)
print(f"Embedding shape: {test_embeddings.shape}")
print(test_embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (2, 384)
[[-3.44772972e-02  3.10232341e-02  6.73494861e-03  2.61090100e-02
  -3.93620580e-02 -1.60302490e-01  6.69240654e-02 -6.44140830e-03
  -4.74504940e-02  1.47589128e-02  7.08753988e-02  5.55276088e-02
   1.91932786e-02 -2.62513887e-02 -1.01095978e-02 -2.69404612e-02
   2.23074388e-02 -2.22266521e-02 -1.49692640e-01 -1.74930878e-02
   7.67624285e-03  5.43523133e-02  3.25442455e-03  3.17259878e-02
  -8.46214592e-02 -2.94060446e-02  5.15956134e-02  4.81239334e-02
  -3.31479032e-03 -5.82791716e-02  4.19692844e-02  2.22107042e-02
   1.28188804e-01 -2.23389380e-02 -1.16563328e-02  6.29283935e-02
  -3.28763053e-02 -9.12260786e-02 -3.11752185e-02  5.26995547e-02
   4.70348112e-02 -8.42029825e-02 -3.00562028e-02 -2.07448658e-02
   9.51780006e-03 -3.72178596e-03  7.34339422e-03  3.93243283e-02
   9.32741091e-02 -3.78855155e-03 -5.27422018e-02 -5.80582321e-02
  -6.86434843e-03  5.28330356e-03  8.28929991e-02  1.93627682e-02
   6.28443807e-03 -1.03306798e-02  9.03241616e-03 

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)

w = WorkspaceClient()

registered_model_name = "att_log_anomaly_catalog.ml.all_minilm_l6_v2"
model_version = "3"
endpoint_name = "all_minilm_l6_v2"

# Check supported workloads
workload_response = w.api_client.do("GET", "/api/2.0/serving-endpoints:workload-configs")
supported_workloads = {
    config["workload_type"]: [size["key"] for size in config.get("workload_sizes", [])]
    for config in workload_response.get("workload_configs", [])
}
print("Supported workloads:", supported_workloads)

# Configure served entity (CPU, Small — sufficient for all-MiniLM-L6-v2 ~80MB)
served_entity = ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=model_version,
    workload_type=ServingModelWorkloadType("CPU"),
    workload_size="Small",
    scale_to_zero_enabled=True,
)
endpoint_config = EndpointCoreConfigInput(served_entities=[served_entity])

# Create or update endpoint
try:
    existing_endpoint = w.serving_endpoints.get(endpoint_name)
except NotFound:
    existing_endpoint = None

if existing_endpoint is None:
    w.serving_endpoints.create_and_wait(name=endpoint_name, config=endpoint_config)
    print(f"Endpoint Created '{endpoint_name}'...")
else:
    w.serving_endpoints.update_config_and_wait(name=endpoint_name, served_entities=[served_entity])
    print(f"Endpoint Updated'{endpoint_name}'...")

Supported workloads: {'CPU': ['Small', 'Medium', 'Large'], 'CPU_MEDIUM': ['Small', 'Medium', 'Large'], 'CPU_LARGE': ['Small', 'Medium', 'Large'], 'GPU_MEDIUM': ['Small', 'Medium', 'Large'], 'MULTIGPU_MEDIUM': ['Small', 'Medium', 'Large'], 'GPU_SMALL': ['Small', 'Medium', 'Large'], 'GPU_MEDIUM_8': ['Small', 'Medium', 'Large'], 'GPU_LARGE': ['Small', 'Medium', 'Large']}
Endpoint Updated'all_minilm_l6_v2'...


In [0]:
response = w.serving_endpoints.query(
    name="all_minilm_l6_v2",
    inputs=["This is a test sentence",
            "Embeddings from a serving endpoint!"],
)

print(response.predictions)

[[0.07155245542526245, 0.06848025321960449, 0.006603303365409374, 0.10176967829465866, 0.0111222630366683, 0.00024534412659704685, 0.01658669300377369, -0.012005402706563473, 0.042088814079761505, 0.04414106160402298, 0.10785840451717377, -0.06899049133062363, -0.00439904211089015, 0.03352140262722969, 0.017190419137477875, -0.032950084656476974, 0.03063458576798439, -0.04599674791097641, -0.05327228084206581, 0.04025144875049591, 0.04043758660554886, 0.03767037391662598, -0.017523257061839104, 0.01689174212515354, 0.007484925910830498, 0.01625245437026024, -0.05224788188934326, 0.00040398899000138044, 0.07939942181110382, -0.014782227575778961, -0.05030818283557892, 0.0027368946466594934, 0.0559978149831295, 0.06447498500347137, 0.019931474700570107, -0.0033759293146431446, 0.042722661048173904, 0.024140680208802223, 0.009568804875016212, 0.011537646874785423, -0.0006791249616071582, -0.12333039194345474, 0.014848555438220501, 0.017071982845664024, 0.017731765285134315, 0.007266889791

In [0]:
import os
import requests
import numpy as np
import pandas as pd
import json

token = token = dbutils.entry_point.getDbutils().notebook().getContext().apiToken().get()


def create_tf_serving_json(data):
    return {'inputs': {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}

def score_model(dataset):
    url = 'https://fevm-att-log-anomaly.cloud.databricks.com/serving-endpoints/all_minilm_l6_v2/invocations'
    headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
    ds_dict = {'dataframe_split': dataset.to_dict(orient='split')} if isinstance(dataset, pd.DataFrame) else create_tf_serving_json(dataset)
    data_json = json.dumps(ds_dict, allow_nan=True)
    response = requests.request(method='POST', headers=headers, url=url, data=data_json)
    if response.status_code != 200:
        raise Exception(f'Request failed with status {response.status_code}, {response.text}')
    return response.json()

score_model(pd.DataFrame([["This is a test sentence",
            "Embeddings from a serving endpoint!"]]))

{'predictions': [[0.07155242562294006,
   0.0684802383184433,
   0.00660336297005415,
   0.10176960378885269,
   0.011122249998152256,
   0.00024540830054320395,
   0.016586659476161003,
   -0.012005331926047802,
   0.042088862508535385,
   0.0441410169005394,
   0.10785844922065735,
   -0.06899050623178482,
   -0.004398956894874573,
   0.0335213877260685,
   0.01719040423631668,
   -0.03295006603002548,
   0.030634569004178047,
   -0.04599670320749283,
   -0.0532723069190979,
   0.04025154933333397,
   0.04043768346309662,
   0.03767038881778717,
   -0.017523368820548058,
   0.016891783103346825,
   0.007484962698072195,
   0.016252439469099045,
   -0.05224783346056938,
   0.0004039549676235765,
   0.079399473965168,
   -0.014782203361392021,
   -0.05030819773674011,
   0.0027369228191673756,
   0.055997852236032486,
   0.06447502225637436,
   0.0199314933270216,
   -0.0033759367652237415,
   0.0427226684987545,
   0.02414068952202797,
   0.009568720124661922,
   0.011537675745785236,

In [0]:
from pyspark.sql import functions as F

# Create a sample Spark DataFrame with text data
data = [
    ("This is an example sentence",),
    ("Each sentence is converted to embeddings",),
    ("Databricks makes ML easy",),
    ("Sentence transformers are powerful",),
]
df = spark.createDataFrame(data, ["text"])

# Use ai_query to call the serving endpoint (serves UC-registered model v3)
df_with_embeddings = df.withColumn(
    "embedding",
    F.expr("ai_query('all_minilm_l6_v2', text, returnType => 'ARRAY<FLOAT>')")
)
display(df_with_embeddings)

text embedding This is an example sentence List(0.06765687, 0.06349589, 0.048713148, 0.07930494, 0.037448034, 0.0026527683, 0.039374966, -0.0070984364, 0.059361435, 0.031537008, 0.060098037, -0.05290518, 0.040606797, -0.025930872, 0.029842805, 0.001126967, 0.07351487, -0.05038195, -0.122386634, 0.023702828, 0.029726481, 0.042476937, 0.025633806, 0.0019952003, -0.056919057, -0.027159786, -0.032903556, 0.06602484, 0.11900708, -0.045879085, -0.072621495, -0.032584008, 0.05234136, 0.04505528, 0.008253014, 0.03670234, -0.01394149, 0.06539188, -0.026427276, 2.0637493E-4, -0.013664403, -0.03628099, -0.019504348, -0.028973812, 0.039427005, -0.08840909, 0.002624212, 0.013671421, 0.048306335, -0.031156573, -0.117329165, -0.051168986, -0.08852875, -0.02189628, 0.014298672, 0.044416785, -0.01348151, 0.07433925, 0.02663828, -0.019876234, 0.017919054, -0.010605257, -0.09042631, 0.021326942, 0.1412049, -0.006471756, -0.0014038574, -0.015361001, -0.08735716, 0.07221743, 0.020140264, 0.042558778, -0.034901414, 3.1960462E-4, -0.080297075, -0.032747235, 0.028526818, -0.05136582, 0.10938917, 0.08193281, -0.098403975, -0.093409516, -0.015129264, 0.04512486, 0.049417146, -0.025186827, 0.015707726, -0.12929076, 0.005318939, 0.0040234574, -0.023457222, -0.06729828, 0.029228065, -0.02608451, 0.013062514, -0.03116632, -0.048271377, -0.055885997, -0.038750548, 0.120010756, -0.010392393, 0.048970457, 0.055353656, 0.044935893, -0.0040096603, -0.102959774, -0.029296868, -0.05834016, 0.02704722, -0.022016952, -0.07222415, -0.041386932, -0.019329831, 0.0027332788, 2.769174E-4, -0.09675877, -0.10057477, -0.0141923055, -0.08078918, 0.045392513, 0.024504088, 0.059761386, -0.07381853, 0.011984359, -0.066340335, -0.076904535, 0.038515754, -5.5936174E-33, 0.028001316, -0.05607844, -0.048660215, 0.021557003, 0.060198102, -0.048140272, -0.035024624, 0.019331405, -0.017515218, -0.038921002, -0.00381061, -0.01702876, 0.028209975, 0.012828983, 0.04716013, 0.062103003, -0.06435887, 0.12928562, -0.013123165, 0.052306943, -0.037368078, 0.028909491, -0.016898073, -0.023733033, -0.03334921, -0.05167626, 0.015535699, 0.020880323, -0.012537161, 0.04595791, 0.037272085, 0.028056683, -0.059000533, -0.011698846, 0.049218215, 0.047032863, 0.07354875, -0.037052967, 0.003984558, 0.01064123, -1.615866E-4, -0.052716587, 0.02759277, -0.039292157, 0.084471755, 0.04868607, -0.004858759, 0.017994829, -0.04285697, 0.012337551, 0.0063996078, 0.04048225, 0.014888759, -0.015394158, 0.07629474, 0.023704357, 0.044523753, 0.050819527, -0.00231249, -0.018873772, -0.012333539, 0.046600237, -0.056343816, 0.062992655, -0.0315535, 0.032491248, 0.023467325, -0.06554375, 0.020170925, 0.025708267, -0.0123868305, -0.008364965, -0.06643777, 0.094307385, -0.03570928, -0.034248345, -0.006663515, -0.0080152415, -0.030971147, 0.04330123, -0.0082139615, -0.15079503, 0.0307692, 0.0400719, -0.03792937, 0.0019321139, 0.04005302, -0.08770743, -0.036849145, 0.008579541, -0.031925146, -0.012525822, 0.07355392, 0.0013473149, 0.020591885, 2.7109787E-33, -0.051857688, 0.057836108, -0.09189854, 0.0394422, 0.10557655, -0.019691242, 0.061840247, -0.076346464, 0.024088066, 0.09400495, -0.116535485, 0.03711987, 0.05224255, -0.003958522, 0.05722148, 0.0053285854, 0.12401681, 0.0139022665, -0.011024965, 0.03560536, -0.03307551, 0.08165736, -0.015200386, 0.060558505, -0.060139727, 0.032610275, -0.034829594, -0.01698814, -0.09749079, -0.02714837, 0.0017471076, -0.076898225, -0.04318583, -0.018998517, -0.029166082, 0.0577488, 0.024182154, -0.011690178, -0.062143486, 0.02843518, -2.375238E-4, -0.025178354, 0.004396335, 0.08128401, 0.036418468, -0.06040066, -0.036551785, -0.079374805, -0.005085305, 0.06696992, -0.11778436, 0.032374367, -0.047125176, -0.013446005, -0.09484455, 0.008249453, -0.010674847, -0.06818821, 0.0011181227, 0.024802014, -0.06358897, 0.0284493, -0.026130369, 0.085811116, 0.114682265, -0.05353452, -0.05635883, 0.04260089, 0.01094543, 0.020957831, 0.10013118, 0.03260515, -0.18420878, -0.039320838, -0.06914543, 

In [0]:
import mlflow
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, FloatType

# Create a sample Spark DataFrame
data = [
    ("This is an example sentence",),
    ("Each sentence is converted to embeddings",),
    ("Databricks makes ML easy",),
    ("Sentence transformers are powerful",),
]
df = spark.createDataFrame(data, ["text"])

# Load the UC-registered model on the driver
uc_model = mlflow.pyfunc.load_model("models:/att_log_anomaly_catalog.ml.all_minilm_l6_v2/3")

# Collect to driver, compute embeddings, convert back to Spark DataFrame
pdf = df.toPandas()
pdf["embedding"] = uc_model.predict(pdf["text"].tolist()).tolist()

df_with_embeddings = spark.createDataFrame(pdf, schema=df.schema.add("embedding", ArrayType(FloatType())))
display(df_with_embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

text embedding This is an example sentence List(0.067656845, 0.0634959, 0.048713107, 0.07930494, 0.037448004, 0.0026528095, 0.039375022, -0.0070984485, 0.059361376, 0.03153704, 0.0600981, -0.052905153, 0.04060677, -0.025930889, 0.02984281, 0.001126944, 0.073514916, -0.05038185, -0.12238666, 0.023702819, 0.02972649, 0.042476848, 0.025633758, 0.0019952205, -0.05691904, -0.027159838, -0.032903556, 0.0660249, 0.11900716, -0.045879055, -0.07262149, -0.03258395, 0.052341394, 0.045055363, 0.008252949, 0.036702413, -0.013941547, 0.065391906, -0.02642723, 2.0637493E-4, -0.013664318, -0.03628096, -0.019504294, -0.028973876, 0.039427094, -0.08840909, 0.0026241883, 0.013671401, 0.048306253, -0.031156553, -0.117329255, -0.05116906, -0.08852884, -0.021896197, 0.014298591, 0.044416815, -0.0134814605, 0.0743392, 0.026638282, -0.019876227, 0.017919071, -0.010605227, -0.0904264, 0.021326995, 0.14120494, -0.0064717294, -0.0014038607, -0.015360985, -0.08735724, 0.072217405, 0.020140324, 0.04255871, -0.03490139, 3.1957327E-4, -0.08029708, -0.03274728, 0.028526798, -0.051365726, 0.109389216, 0.08193281, -0.098404, -0.09340956, -0.015129161, 0.045124866, 0.0494172, -0.02518677, 0.015707755, -0.12929074, 0.0053188587, 0.004023459, -0.02345723, -0.06729839, 0.029228076, -0.026084533, 0.0130625395, -0.03116631, -0.04827141, -0.055885922, -0.038750485, 0.12001074, -0.010392412, 0.04897048, 0.05535372, 0.044935945, -0.004009736, -0.10295965, -0.029296903, -0.05834019, 0.027047254, -0.022016944, -0.072224125, -0.041386977, -0.019329822, 0.0027331857, 2.7701346E-4, -0.09675879, -0.10057477, -0.014192252, -0.08078915, 0.045392506, 0.024504103, 0.05976133, -0.07381855, 0.011984354, -0.066340394, -0.07690443, 0.038515735, -5.593621E-33, 0.028001405, -0.05607848, -0.04866017, 0.021557052, 0.060198043, -0.048140254, -0.035024654, 0.019331355, -0.017515153, -0.038921, -0.003810645, -0.017028783, 0.02820999, 0.012829001, 0.047160115, 0.062102955, -0.06435889, 0.12928563, -0.013123054, 0.052306972, -0.03736815, 0.028909428, -0.016898045, -0.023733122, -0.03334907, -0.051676326, 0.015535609, 0.02088028, -0.012537138, 0.04595792, 0.03727207, 0.02805673, -0.0590005, -0.011698837, 0.049218245, 0.047032915, 0.073548675, -0.037052993, 0.0039846594, 0.0106412005, -1.615091E-4, -0.052716617, 0.027592795, -0.039292123, 0.08447183, 0.04868609, -0.004858764, 0.017994896, -0.04285698, 0.0123375105, 0.0063996273, 0.04048228, 0.014888794, -0.015394221, 0.07629481, 0.023704384, 0.044523705, 0.05081963, -0.0023124875, -0.018873708, -0.01233351, 0.046600252, -0.056343786, 0.062992655, -0.031553525, 0.032491297, 0.023467276, -0.06554379, 0.020171005, 0.025708213, -0.012386857, -0.008364977, -0.06643776, 0.09430736, -0.035709295, -0.034248326, -0.0066635576, -0.008015286, -0.030971153, 0.043301255, -0.008214031, -0.15079509, 0.030769268, 0.04007187, -0.037929446, 0.0019321061, 0.040053055, -0.08770738, -0.036849104, 0.008579451, -0.03192524, -0.012525756, 0.07355382, 0.0013473266, 0.020591902, 2.7109822E-33, -0.051857773, 0.05783608, -0.091898546, 0.03944221, 0.105576485, -0.019691266, 0.06184022, -0.07634651, 0.02408802, 0.09400485, -0.11653547, 0.037119847, 0.052242525, -0.003958534, 0.057221405, 0.0053285873, 0.124016866, 0.013902209, -0.011024969, 0.035605393, -0.033075437, 0.081657395, -0.015200435, 0.06055847, -0.060139686, 0.032610275, -0.034829658, -0.016988222, -0.097490765, -0.027148418, 0.0017471559, -0.07689826, -0.043185797, -0.01899844, -0.029166153, 0.0577488, 0.024182122, -0.011690229, -0.062143564, 0.028435167, -2.3749413E-4, -0.025178328, 0.004396317, 0.081284, 0.036418438, -0.06040058, -0.03655183, -0.07937477, -0.005085283, 0.06696997, -0.11778435, 0.032374334, -0.047125254, -0.013446003, -0.094844535, 0.008249511, -0.01067487, -0.06818829, 0.0011181843, 0.024801979, -0.063588984, 0.028449295, -0.026130361, 0.08581112, 0.11468236, -0.05353453, -0.056358844, 0.04260086, 0.010945431, 0.020957854, 0.10013121, 0.032605153, -0.18420878, -0.039320886, -0.06914552, -0.0

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from databricks.sdk import WorkspaceClient

# Create a sample Spark DataFrame
data = [
    ("This is an example sentence",),
    ("Each sentence is converted to embeddings",),
    ("Databricks makes ML easy",),
    ("Sentence transformers are powerful",),
]
df = spark.createDataFrame(data, ["text"])


@pandas_udf("array<float>")
def score_endpoint(texts: pd.Series) -> pd.Series:
    """Score a batch of texts by calling the serving endpoint."""
    w = WorkspaceClient()  # Ensure proper authentication configuration, possibly by setting environment variables or configuration files
    response = w.serving_endpoints.query(
        name="all_minilm_l6_v2",
        inputs=texts.tolist(),
    )
    return pd.Series([emb for emb in response.predictions])


df_scored = df.withColumn("embedding", score_endpoint(F.col("text")))
display(df_scored)

In [0]:
import mlflow
import packaging.version

# Workaround: serverless reports runtime as '18.x-photon-scala2' which isn't PEP 440 compliant.
# Patch __init__ directly so all existing references to Version (including mlflow's) use the fix.
_orig_version_init = packaging.version.Version.__init__

def _safe_version_init(self, version):
    if isinstance(version, str) and ("photon" in version or "scala" in version):
        version = version.split(".")[0] + ".0"
    _orig_version_init(self, version)

packaging.version.Version.__init__ = _safe_version_init

try:
    predict_udf = mlflow.pyfunc.spark_udf(
        spark,
        model_uri="models:/att_log_anomaly_catalog.ml.all_minilm_l6_v2/3",
        result_type="array<float>",
    )
    df_scored = df.withColumn("embedding", predict_udf("text"))
    display(df_scored)
finally:
    packaging.version.Version.__init__ = _orig_version_init

2026/08/19 05:56:12 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2026/08/19 05:56:12 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


text embedding This is an example sentence List(0.06765687, 0.06349589, 0.048713148, 0.07930494, 0.037448034, 0.0026527683, 0.039374966, -0.0070984364, 0.059361435, 0.031537008, 0.060098037, -0.05290518, 0.040606797, -0.025930872, 0.029842805, 0.001126967, 0.07351487, -0.05038195, -0.122386634, 0.023702828, 0.029726481, 0.042476937, 0.025633806, 0.0019952003, -0.056919057, -0.027159786, -0.032903556, 0.06602484, 0.11900708, -0.045879085, -0.072621495, -0.032584008, 0.05234136, 0.04505528, 0.008253014, 0.03670234, -0.01394149, 0.06539188, -0.026427276, 2.0637493E-4, -0.013664403, -0.03628099, -0.019504348, -0.028973812, 0.039427005, -0.08840909, 0.002624212, 0.013671421, 0.048306335, -0.031156573, -0.117329165, -0.051168986, -0.08852875, -0.02189628, 0.014298672, 0.044416785, -0.01348151, 0.07433925, 0.02663828, -0.019876234, 0.017919054, -0.010605257, -0.09042631, 0.021326942, 0.1412049, -0.006471756, -0.0014038574, -0.015361001, -0.08735716, 0.07221743, 0.020140264, 0.042558778, -0.034901414, 3.1960462E-4, -0.080297075, -0.032747235, 0.028526818, -0.05136582, 0.10938917, 0.08193281, -0.098403975, -0.093409516, -0.015129264, 0.04512486, 0.049417146, -0.025186827, 0.015707726, -0.12929076, 0.005318939, 0.0040234574, -0.023457222, -0.06729828, 0.029228065, -0.02608451, 0.013062514, -0.03116632, -0.048271377, -0.055885997, -0.038750548, 0.120010756, -0.010392393, 0.048970457, 0.055353656, 0.044935893, -0.0040096603, -0.102959774, -0.029296868, -0.05834016, 0.02704722, -0.022016952, -0.07222415, -0.041386932, -0.019329831, 0.0027332788, 2.769174E-4, -0.09675877, -0.10057477, -0.0141923055, -0.08078918, 0.045392513, 0.024504088, 0.059761386, -0.07381853, 0.011984359, -0.066340335, -0.076904535, 0.038515754, -5.5936174E-33, 0.028001316, -0.05607844, -0.048660215, 0.021557003, 0.060198102, -0.048140272, -0.035024624, 0.019331405, -0.017515218, -0.038921002, -0.00381061, -0.01702876, 0.028209975, 0.012828983, 0.04716013, 0.062103003, -0.06435887, 0.12928562, -0.013123165, 0.052306943, -0.037368078, 0.028909491, -0.016898073, -0.023733033, -0.03334921, -0.05167626, 0.015535699, 0.020880323, -0.012537161, 0.04595791, 0.037272085, 0.028056683, -0.059000533, -0.011698846, 0.049218215, 0.047032863, 0.07354875, -0.037052967, 0.003984558, 0.01064123, -1.615866E-4, -0.052716587, 0.02759277, -0.039292157, 0.084471755, 0.04868607, -0.004858759, 0.017994829, -0.04285697, 0.012337551, 0.0063996078, 0.04048225, 0.014888759, -0.015394158, 0.07629474, 0.023704357, 0.044523753, 0.050819527, -0.00231249, -0.018873772, -0.012333539, 0.046600237, -0.056343816, 0.062992655, -0.0315535, 0.032491248, 0.023467325, -0.06554375, 0.020170925, 0.025708267, -0.0123868305, -0.008364965, -0.06643777, 0.094307385, -0.03570928, -0.034248345, -0.006663515, -0.0080152415, -0.030971147, 0.04330123, -0.0082139615, -0.15079503, 0.0307692, 0.0400719, -0.03792937, 0.0019321139, 0.04005302, -0.08770743, -0.036849145, 0.008579541, -0.031925146, -0.012525822, 0.07355392, 0.0013473149, 0.020591885, 2.7109787E-33, -0.051857688, 0.057836108, -0.09189854, 0.0394422, 0.10557655, -0.019691242, 0.061840247, -0.076346464, 0.024088066, 0.09400495, -0.116535485, 0.03711987, 0.05224255, -0.003958522, 0.05722148, 0.0053285854, 0.12401681, 0.0139022665, -0.011024965, 0.03560536, -0.03307551, 0.08165736, -0.015200386, 0.060558505, -0.060139727, 0.032610275, -0.034829594, -0.01698814, -0.09749079, -0.02714837, 0.0017471076, -0.076898225, -0.04318583, -0.018998517, -0.029166082, 0.0577488, 0.024182154, -0.011690178, -0.062143486, 0.02843518, -2.375238E-4, -0.025178354, 0.004396335, 0.08128401, 0.036418468, -0.06040066, -0.036551785, -0.079374805, -0.005085305, 0.06696992, -0.11778436, 0.032374367, -0.047125176, -0.013446005, -0.09484455, 0.008249453, -0.010674847, -0.06818821, 0.0011181227, 0.024802014, -0.06358897, 0.0284493, -0.026130369, 0.085811116, 0.114682265, -0.05353452, -0.05635883, 0.04260089, 0.01094543, 0.020957831, 0.10013118, 0.03260515, -0.18420878, -0.039320838, -0.06914543, 